# HSE DSBA Chatbot — RAG Retrieval Evaluation
Тестирование качества ретрива на 50 вопросах по программе ПАД

## 1. Импорты

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import re

## 2. Загрузка и разбивка документа на чанки

In [10]:
with open('About-program.txt', 'r', encoding='utf-8') as f:
    text = f.read()

def split_into_chunks(text, chunk_size=500, overlap=50):
    sections = re.split(r'---+', text)
    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        words = section.split()
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if len(chunk) > 50:
                chunks.append(chunk)
    return chunks

chunks = split_into_chunks(text)
print(f'Всего чанков: {len(chunks)}')
print('\nПример первого чанка:')
print(chunks[0][:300])

Всего чанков: 67

Пример первого чанка:
ВСЯ ПОСЛЕДУЮЩАЯ ИНФОРМАЦИЯ БУДЕТ АКТУАЛЬНАЯ НА 2024/2025 ГОД, НЕКОТОРЫЕ ДАННЫЕ БУДУТ АКТУАЛЬНЫ НА 2025/2026 УЧЕБНЫЙ ГОД: О программе: Целью программы является подготовка высококвалифицированных аналитиков и специалистов в области Data Scienсе, обладающих пониманием задач экономики и финансов, бизнес


## 3. Создание ChromaDB и загрузка чанков

In [11]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

client = chromadb.Client()
collection = client.create_collection(name='dsba_docs')

embeddings = model.encode(chunks).tolist()

collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f'chunk_{i}' for i in range(len(chunks))]
)

print(f'Загружено {collection.count()} чанков в ChromaDB')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Загружено 67 чанков в ChromaDB


## 4. Золотой набор — 50 вопросов

In [12]:
golden_set = [
    ('Какова основная цель программы ПАД?', 'подготовка высококвалифицированных аналитиков'),
    ('В каком году была создана программа ПАД?', '2018'),
    ('С каким британским университетом создана программа?', 'LSE'),
    ('На каком факультете реализуется программа ПАД?', 'факультете компьютерных наук'),
    ('В каком году ВШЭ и Яндекс открыли факультет компьютерных наук?', '2014'),
    ('Какие специализации есть на программе?', 'Анализ данных в бизнесе'),
    ('На каком языке ведётся обучение на программе?', 'английском'),
    ('Когда был выпущен первый набор студентов ПАД?', '2022'),
    ('Какие предметы ЕГЭ нужны для поступления на ПАД?', 'Математика'),
    ('Какой минимальный балл ЕГЭ по математике?', '70'),
    ('Какой минимальный балл ЕГЭ по физике или информатике?', '60'),
    ('Какой минимальный балл ЕГЭ по русскому языку?', '60'),
    ('Сколько платных мест на программе ПАД?', '220'),
    ('Есть ли бюджетные места на программе ПАД?', 'бюджетные места не предусмотрены'),
    ('Какова стоимость обучения на 2025 год?', '1 млн'),
    ('Имеют ли победители олимпиад льготы при поступлении?', 'олимпиад имеют право'),
    ('Какой коэффициент у балла ЕГЭ по математике при ранжировании?', 'коэффициент 3'),
    ('До какой даты нужно предоставить диплом олимпиады для скидки?', '22 июля'),
    ('Сколько мест для иностранных студентов?', '10 платных мест'),
    ('Какие экзамены сдают иностранцы по отдельному конкурсу?', 'Математику и Английский'),
    ('Какой минимальный балл по математике для иностранцев?', '75'),
    ('Какой минимальный балл по английскому для иностранцев?', '70'),
    ('Можно ли сдать вступительные экзамены дистанционно?', 'дистанционно'),
    ('Какой экзамен нужен иностранцу с российским гражданством?', 'русскому'),
    ('Когда публикуется список претендентов на скидку?', '25 июля'),
    ('Какой максимальный размер скидки по результатам обучения?', '75%'),
    ('Какой минимальный размер скидки по результатам обучения?', '10%'),
    ('Кому адресовать вопросы по скидкам?', 'Шахвердян'),
    ('Учитываются ли индивидуальные достижения при ранжировании?', 'индивидуальные достижения'),
    ('По каким олимпиадам учитываются достижения для скидки?', 'математике, информатике, физике'),
    ('На каком курсе начинается изучение Python?', 'Python'),
    ('На каком курсе изучается Машинное обучение?', 'Машинное обучение'),
    ('На каком курсе изучаются Базы данных?', 'Базы данных'),
    ('На каком курсе изучается Глубокое обучение?', 'Глубокое обучение'),
    ('На каком курсе изучается Эконометрика?', 'Эконометрика'),
    ('Является ли курс Линейная алгебра обязательным?', 'Линейная алгебра'),
    ('На каком курсе изучается Компьютерное зрение?', 'Компьютерное зрение'),
    ('На каком курсе изучается Анализ временных рядов?', 'Анализ временных рядов'),
    ('Можно ли получить диплом LSE обучаясь на ПАД?', 'приостановку сотрудничества'),
    ('Чем ПАД отличается от ПМИ?', 'ПМИ'),
    ('Чем ПАД отличается от ЭиАД?', 'ЭиАД'),
    ('Предоставляется ли отсрочка от армии студентам ПАД?', 'отсрочка'),
    ('Трудно ли сразу учиться на английском?', 'языковой подготовке'),
    ('Когда на ПАД начинается практическая проектная работа?', '2-го курса'),
    ('Есть ли на ФКН военный учебный центр?', 'Военный учебный центр'),
    ('Изменился ли учебный план после приостановки сотрудничества с LSE?', 'изменений в учебном плане не произошло'),
    ('Какой размер стипендии Яндекса для студентов-бакалавров?', '30'),
    ('Сколько студентов-бакалавров получают стипендию Яндекса?', 'десять'),
    ('В каких компаниях могут работать выпускники ПАД?', 'Яндекс'),
    ('Какую магистерскую программу открыл ФКН в партнёрстве со Сбером?', 'Финансовые технологии'),
]

print(f'Золотой набор: {len(golden_set)} вопросов')

Золотой набор: 50 вопросов


## 5. Оценка качества ретрива — Hit Rate@3 и Hit Rate@5

In [13]:
def check_hit(retrieved_chunks, keyword):
    keyword_lower = keyword.lower()
    for chunk in retrieved_chunks:
        if keyword_lower in chunk.lower():
            return True
    return False

results = []

for question, keyword in golden_set:
    q_embedding = model.encode([question]).tolist()
    response = collection.query(
        query_embeddings=q_embedding,
        n_results=5
    )
    retrieved = response['documents'][0]
    hit3 = check_hit(retrieved[:3], keyword)
    hit5 = check_hit(retrieved[:5], keyword)
    results.append({
        'Вопрос': question,
        'Ключевое слово': keyword,
        'Hit@3': '✅' if hit3 else '❌',
        'Hit@5': '✅' if hit5 else '❌'
    })

df = pd.DataFrame(results)
hit3_rate = (df['Hit@3'] == '✅').mean()
hit5_rate = (df['Hit@5'] == '✅').mean()

print(f'Hit Rate@3: {hit3_rate:.1%}')
print(f'Hit Rate@5: {hit5_rate:.1%}')
print(f'Промахи@3: {(df["Hit@3"] == "❌").sum()} вопросов из {len(df)}')
df

Hit Rate@3: 52.0%
Hit Rate@5: 64.0%
Промахи@3: 24 вопросов из 50


,Вопрос,Ключевое слово,Hit@3,Hit@5
0,Какова основная цель программы ПАД?,подготовка высококвалифицированных аналитиков,✅,✅
1,В каком году была создана программа ПАД?,2018,✅,✅
2,С каким британским университетом создана прогр...,LSE,✅,✅
3,На каком факультете реализуется программа ПАД?,факультете компьютерных наук,❌,✅
4,В каком году ВШЭ и Яндекс открыли факультет ко...,2014,❌,✅
5,Какие специализации есть на программе?,Анализ данных в бизнесе,❌,❌
6,На каком языке ведётся обучение на программе?,английском,❌,❌
7,Когда был выпущен первый набор студентов ПАД?,2022,✅,✅
8,Какие предметы ЕГЭ нужны для поступления на ПАД?,Математика,✅,✅
9,Какой минимальный балл ЕГЭ по математике?,70,❌,❌


In [14]:
client2 = chromadb.Client()
collection2 = client2.create_collection(name='dsba_docs_small')

chunks_small = split_into_chunks(text, chunk_size=150, overlap=30)
embeddings_small = model.encode(chunks_small).tolist()

collection2.add(
    documents=chunks_small,
    embeddings=embeddings_small,
    ids=[f'chunk_{i}' for i in range(len(chunks_small))]
)

print(f'Чанков с новым размером: {len(chunks_small)}')

results2 = []
for question, keyword in golden_set:
    q_embedding = model.encode([question]).tolist()
    response = collection2.query(query_embeddings=q_embedding, n_results=5)
    retrieved = response['documents'][0]
    hit3 = check_hit(retrieved[:3], keyword)
    hit5 = check_hit(retrieved[:5], keyword)
    results2.append({
        'Вопрос': question,
        'Ключевое слово': keyword,
        'Hit@3': '✅' if hit3 else '❌',
        'Hit@5': '✅' if hit5 else '❌'
    })

df2 = pd.DataFrame(results2)
hit3_rate2 = (df2['Hit@3'] == '✅').mean()
hit5_rate2 = (df2['Hit@5'] == '✅').mean()

print(f'Hit Rate@3: {hit3_rate2:.1%}')
print(f'Hit Rate@5: {hit5_rate2:.1%}')
print(f'Промахи@3: {(df2["Hit@3"] == "❌").sum()} вопросов из 50')


Чанков с новым размером: 81
Hit Rate@3: 62.0%
Hit Rate@5: 64.0%
Промахи@3: 19 вопросов из 50


In [15]:
client3 = chromadb.Client()
collection3 = client3.create_collection(name='dsba_docs_tiny')

chunks_tiny = split_into_chunks(text, chunk_size=80, overlap=20)
embeddings_tiny = model.encode(chunks_tiny).tolist()

collection3.add(
    documents=chunks_tiny,
    embeddings=embeddings_tiny,
    ids=[f'chunk_{i}' for i in range(len(chunks_tiny))]
)

print(f'Чанков с новым размером: {len(chunks_tiny)}')

results3 = []
for question, keyword in golden_set:
    q_embedding = model.encode([question]).tolist()
    response = collection3.query(query_embeddings=q_embedding, n_results=5)
    retrieved = response['documents'][0]
    hit3 = check_hit(retrieved[:3], keyword)
    hit5 = check_hit(retrieved[:5], keyword)
    results3.append({
        'Вопрос': question,
        'Ключевое слово': keyword,
        'Hit@3': '✅' if hit3 else '❌',
        'Hit@5': '✅' if hit5 else '❌'
    })

df3 = pd.DataFrame(results3)
hit3_rate3 = (df3['Hit@3'] == '✅').mean()
hit5_rate3 = (df3['Hit@5'] == '✅').mean()

print(f'Hit Rate@3: {hit3_rate3:.1%}')
print(f'Hit Rate@5: {hit5_rate3:.1%}')
print(f'Промахи@3: {(df3["Hit@3"] == "❌").sum()} вопросов из 50')


Чанков с новым размером: 99
Hit Rate@3: 68.0%
Hit Rate@5: 72.0%
Промахи@3: 16 вопросов из 50


In [16]:
client4 = chromadb.Client()
collection4 = client4.create_collection(name='dsba_docs_micro')

chunks_micro = split_into_chunks(text, chunk_size=40, overlap=10)
embeddings_micro = model.encode(chunks_micro).tolist()

collection4.add(
    documents=chunks_micro,
    embeddings=embeddings_micro,
    ids=[f'chunk_{i}' for i in range(len(chunks_micro))]
)

print(f'Чанков с новым размером: {len(chunks_micro)}')

results4 = []
for question, keyword in golden_set:
    q_embedding = model.encode([question]).tolist()
    response = collection4.query(query_embeddings=q_embedding, n_results=5)
    retrieved = response['documents'][0]
    hit3 = check_hit(retrieved[:3], keyword)
    hit5 = check_hit(retrieved[:5], keyword)
    results4.append({
        'Вопрос': question,
        'Ключевое слово': keyword,
        'Hit@3': '✅' if hit3 else '❌',
        'Hit@5': '✅' if hit5 else '❌'
    })

df4 = pd.DataFrame(results4)
hit3_rate4 = (df4['Hit@3'] == '✅').mean()
hit5_rate4 = (df4['Hit@5'] == '✅').mean()

print(f'Hit Rate@3: {hit3_rate4:.1%}')
print(f'Hit Rate@5: {hit5_rate4:.1%}')
print(f'Промахи@3: {(df4["Hit@3"] == "❌").sum()} вопросов из 50')


Чанков с новым размером: 141
Hit Rate@3: 62.0%
Hit Rate@5: 66.0%
Промахи@3: 19 вопросов из 50


## 6. Анализ промахов

In [17]:
misses = df[df['Hit@3'] == '❌']
print(f'Вопросы без попадания в топ-3 ({len(misses)} шт.):\n')
for _, row in misses.iterrows():
    print(f"  - {row['Вопрос']}")
    print(f"    (искали: '{row['Ключевое слово']}')\n")

Вопросы без попадания в топ-3 (24 шт.):

  - На каком факультете реализуется программа ПАД?
    (искали: 'факультете компьютерных наук')

  - В каком году ВШЭ и Яндекс открыли факультет компьютерных наук?
    (искали: '2014')

  - Какие специализации есть на программе?
    (искали: 'Анализ данных в бизнесе')

  - На каком языке ведётся обучение на программе?
    (искали: 'английском')

  - Какой минимальный балл ЕГЭ по математике?
    (искали: '70')

  - Какой минимальный балл ЕГЭ по русскому языку?
    (искали: '60')

  - Какой коэффициент у балла ЕГЭ по математике при ранжировании?
    (искали: 'коэффициент 3')

  - Какие экзамены сдают иностранцы по отдельному конкурсу?
    (искали: 'Математику и Английский')

  - Какой минимальный балл по математике для иностранцев?
    (искали: '75')

  - Какой минимальный балл по английскому для иностранцев?
    (искали: '70')

  - Какой минимальный размер скидки по результатам обучения?
    (искали: '10%')

  - На каком курсе изучается Машинное 

In [18]:
misses_best = df3[df3['Hit@3'] == '❌']
print(f'Промахи оптимального варианта (chunk=80) — {len(misses_best)} шт.:\n')
for _, row in misses_best.iterrows():
    print(f"  - {row['Вопрос']}")
    print(f"    (искали: '{row['Ключевое слово']}')\n")


Промахи оптимального варианта (chunk=80) — 16 шт.:

  - На каком факультете реализуется программа ПАД?
    (искали: 'факультете компьютерных наук')

  - Какие специализации есть на программе?
    (искали: 'Анализ данных в бизнесе')

  - На каком языке ведётся обучение на программе?
    (искали: 'английском')

  - Какой минимальный балл ЕГЭ по физике или информатике?
    (искали: '60')

  - Имеют ли победители олимпиад льготы при поступлении?
    (искали: 'олимпиад имеют право')

  - Какие экзамены сдают иностранцы по отдельному конкурсу?
    (искали: 'Математику и Английский')

  - Какой минимальный балл по английскому для иностранцев?
    (искали: '70')

  - Какой минимальный размер скидки по результатам обучения?
    (искали: '10%')

  - На каком курсе изучается Машинное обучение?
    (искали: 'Машинное обучение')

  - На каком курсе изучаются Базы данных?
    (искали: 'Базы данных')

  - На каком курсе изучается Глубокое обучение?
    (искали: 'Глубокое обучение')

  - На каком курс

## 7. Сохранение результатов

In [ ]:
df.to_csv('retrieval_results.csv', index=False, encoding='utf-8-sig')
print('Результаты сохранены в retrieval_results.csv')
print('\n=== ИТОГОВАЯ СВОДКА ===')
print(f'Всего вопросов: {len(df)}')
print(f'Hit Rate@3:     {hit3_rate:.1%}')
print(f'Hit Rate@5:     {hit5_rate:.1%}')

In [19]:
df3.to_csv('retrieval_results_best.csv', index=False, encoding='utf-8-sig')
print('Сохранено в retrieval_results_best.csv')


Сохранено в retrieval_results_best.csv
